# GOAL: Build SARIMAX model

Note: using "previous day" would NOT mean that today is used to predict tomorrow. It means the prediction for today is used to predict tomorrow.

In [1]:
#import statements

#import everything!
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, datetime, timedelta
from itertools import product
from PreRun import PreRun, PostRun
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX



#path to data
read_path = "../../../../data_ds_project/parquet_cleaned_energy"
#systems
good_systems_list = [4, 10, 33, 36, 50, 51, 1199, 1204, 1283, 1284, 1289, 1332, 4902, 4903]
reader_types = ["meter", "inverter", None]
#systems_cleaned
systems_cleaned = pd.read_csv("../../../data/core/systems_cleaned.csv")

In [2]:
def use_sarimax(df_train, df_ho, p=0, d=0, q=0, P=0, D=0, Q=0, s=0):

    #separate out the exogenous variables
    df_train = df_train.set_index('time')
    df_ho = df_ho.set_index('time')

    # Remove duplicate indices (keep first occurrence)
    df_train = df_train[~df_train.index.duplicated(keep='first')]
    df_ho = df_ho[~df_ho.index.duplicated(keep='first')]

    df_train = df_train.asfreq('h')
    df_ho = df_ho.asfreq('h')

    energy_train = df_train['energy']
    #energy_ho = df_ho['energy']

    exog_train = df_train.drop(columns=['energy'])
    exog_ho = df_ho.drop(columns=['energy'])

    #fit the model
    model_sarimax = SARIMAX(energy_train, exog=exog_train, order=(p,d,q), seasonal_order = (P,D,Q,s)).fit(maxiter=600, disp=False)
    #predict
    y_pred = model_sarimax.forecast(len(df_ho), exog=exog_ho) 

    #error = PostRun.custom_error(energy_ho, y_pred, 1,2)

    return y_pred#, error

In [3]:
# system_reader_pairs = [(4,None),(10,None), (33, None), (50,None), (51,None), (1283,'inverter'),(1283,'meter')]
# #system_reader_pairs = [(1283,'inverter'),(1283,'meter')]

# # p_choices =[2,3]
# # d_choices = [0,1]
# # q_choices = [0,1,2]

# # P_choices = [0,1,2,3]
# # D_choices = [0]
# # Q_choices = [0,1,2]

# p_choices =[2]
# d_choices = [1]
# q_choices = [0]

# P_choices = [0]
# D_choices = [0]
# Q_choices = [0]


# def process_system_pair(pair, p_choices, d_choices, q_choices, P_choices, D_choices, Q_choices, 
#                         read_path, systems_cleaned):
#     """Process a single system_reader_pair and return results"""
#     system_id = pair[0]
#     reader_type = pair[1]
#     print(f'starting system {system_id}, {reader_type}')

#     prerun = PreRun(system_id = system_id, meter_or_inverter = reader_type, path=read_path, systems_cleaned=systems_cleaned)
#     prerun.load_data()
        
#     prerun.fill_missing_hours()
#     prerun.add_energy_features_only(include_last_year = True, remove_last_year_nans=True)
#     print(prerun.amended_data)

#     prerun.add_weather_features_only()

#     all_data=prerun.amended_data

#     prerun.good_end_days_naive(10)
#     prerun.tts_of_data_using_end_days()

#     system_recorded_max = prerun.data['energy'].max()

#     # Build results as a dictionary to avoid pandas reindexing issues
#     all_errors_dict = {}
    
#     for p,d,q,P,D,Q in product(p_choices,d_choices,q_choices,P_choices,D_choices,Q_choices):
#         print(f'     starting p={p}, d={d}, q={q}, P={P}, D={D}, Q={Q}')
#         errors=[]
#         for pred_date in prerun.train_dates['date']:
#             #print(type(date))

        
#             train_data = all_data.loc[(all_data['time'] < pred_date - timedelta(days=1)) &
#                                     (all_data['time'] >= pred_date - timedelta(days=10))].reset_index(drop=True)
#             ho_data = all_data[(all_data['time'] >= pred_date - timedelta(days=1)) 
#                                 & (all_data['time'] <= pred_date)].reset_index(drop=True)

#             y_pred = use_sarimax(train_data, ho_data, p=p, d=d, q=q, P=P, s=0, D=D, Q=Q)
#             energy_ho = ho_data['energy']

#             #make sure value between 0 and highest observed max
#             y_pred = np.clip(y_pred, 0, system_recorded_max)
#             #make sure that if it's not sunlight hours and irradiance = 0, then energy = 0
#             darkness_mask = (~((ho_data['proportion_daytime']==0) & (ho_data['global_tilted_irradiance']==0))).astype(int) # = 0 when both sunlight prop and irr are 0
#             y_pred = y_pred*np.array(darkness_mask)

#             error = PostRun.custom_error(energy_ho, y_pred, 1,2)

#             errors.append(error)

#         # Store error list in dictionary (avoids reindexing issues)
#         all_errors_dict[f'{p},{d},{q},{P},{D},{Q}'] = errors

#     # Convert dictionary to DataFrame at the end (single operation)
#     all_errors_df = pd.DataFrame(all_errors_dict)

#     # # Old code (commented out):
#     # all_errors_df = pd.DataFrame()
#     # 
#     # for p,d,q,P,D,Q in product(p_choices,d_choices,q_choices,P_choices,D_choices,Q_choices):
#     #     print(f'     starting p={p}, d={d}, q={q}, P={P}, D={D}, Q={Q}')
#     #     errors=[]
#     #     for pred_date in prerun.train_dates['date']:
#     #         train_data = all_data.loc[(all_data['time'] < pred_date - timedelta(days=1)) &
#     #                                 (all_data['time'] >= pred_date - timedelta(days=10))].reset_index(drop=True)
#     #         ho_data = all_data[(all_data['time'] >= pred_date - timedelta(days=1)) 
#     #                             & (all_data['time'] <= pred_date)].reset_index(drop=True)
#     #         y_pred = use_sarimax(train_data, ho_data, p=p, d=d, q=q, P=P, s=24, D=D, Q=Q)
#     #         energy_ho = ho_data['energy']
#     #         y_pred = np.clip(y_pred, 0, system_recorded_max)
#     #         darkness_mask = (~((ho_data['proportion_daytime']==0) & (ho_data['global_tilted_irradiance']==0))).astype(int)
#     #         y_pred = y_pred*np.array(darkness_mask)
#     #         error = PostRun.custom_error(energy_ho, y_pred, 1,2)
#     #         errors.append(error)
#     #     all_errors_df[f'{p},{d},{q},{P},{D},{Q}'] = pd.Series(errors)

#     # Save results for this pair
#     output_path = Path(f'sarimax_errors/{system_id}_{reader_type}_sarimax_errors.csv')
#     output_path.parent.mkdir(parents=True, exist_ok=True)
#     all_errors_df.to_csv(output_path)
    
#     return (system_id, reader_type, all_errors_df)


# # # Serial execution (original code, commented out)
# # for pair in system_reader_pairs:
# #     system_id = pair[0]
# #     reader_type = pair[1]
# #     print(f'starting system {system_id}, {reader_type}')

# #     prerun = PreRun(system_id = system_id, meter_or_inverter = reader_type, path=read_path, systems_cleaned=systems_cleaned)
# #     prerun.load_data()
# #         
# #     prerun.fill_missing_hours()
# #     prerun.add_energy_features_only(include_last_year = True, remove_last_year_nans=True, highest_fourier_term_hour=2)
# #     print(prerun.amended_data)

# #     prerun.add_weather_features_only()

# #     all_data=prerun.amended_data

# #     prerun.good_end_days_naive(10)
# #     prerun.tts_of_data_using_end_days()

# #     system_recorded_max = prerun.data['energy'].max()

# #     all_errors_df = pd.DataFrame()
# #     
# #     for p,d,q,P,D,Q in product(p_choices,d_choices,q_choices,P_choices,D_choices,Q_choices):
# #         print(f'     starting p={p}, d={d}, q={q}, P={P}, D={D}, Q={Q}')
# #         errors=[]
# #         for pred_date in prerun.train_dates['date']:
# #             #print(type(date))

# #         
# #             train_data = all_data.loc[(all_data['time'] < pred_date - timedelta(days=1)) &
# #                                     (all_data['time'] >= pred_date - timedelta(days=10))].reset_index(drop=True)
# #             ho_data = all_data[(all_data['time'] >= pred_date - timedelta(days=1)) 
# #                                 & (all_data['time'] <= pred_date)].reset_index(drop=True)

# #             y_pred = use_sarimax(train_data, ho_data, p=p, d=d, q=q, P=P, s=0, D=D, Q=Q)
# #             energy_ho = ho_data['energy']
# #             error = PostRun.custom_error(energy_ho, y_pred, 1,2)

# #             errors.append(error)

# #         all_errors_df[f'{p},{d},{q},{P},{D},{Q}'] = pd.Series(errors)

# # all_errors_df.to_csv(f'sarimax_errors/{system_id}_{reader_type}_sarimax_errors')


# # Parallel execution
# results = Parallel(n_jobs=-1)(
#     delayed(process_system_pair)(
#         pair, 
#         p_choices, 
#         d_choices, 
#         q_choices, 
#         P_choices, 
#         D_choices, 
#         Q_choices, 
#         read_path, 
#         systems_cleaned
#     ) 
#     for pair in system_reader_pairs
# )

# print("\nAll processing complete!")
# for system_id, reader_type, errors_df in results:
#     print(f'Completed: system {system_id}, {reader_type}')

In [3]:
#system_reader_pairs = [(4,None),(10,None), (33, None), (50,None), (51,None), (1283,'inverter'),(1283,'meter')]
system_reader_pairs = [(50,None), (51,None)]

p_choices =[2,3]
#d_choices = [0,1]
d_choices = [0]
q_choices = [0,1]

for pair in system_reader_pairs:
    system_id = pair[0]
    reader_type = pair[1]
    print(f'starting system {system_id}, {reader_type}')

    prerun = PreRun(system_id = system_id, meter_or_inverter = reader_type, path=read_path, systems_cleaned=systems_cleaned)
    prerun.load_data()
        
    prerun.fill_missing_hours()
    prerun.add_energy_features_only(include_last_year = True, remove_last_year_nans=True, highest_fourier_term_hour=2)
    #print(prerun.amended_data)

    prerun.add_weather_features_only()

    all_data=prerun.amended_data

    prerun.good_end_days_naive(10)
    prerun.tts_of_data_using_end_days(remove_first_year=False)

    system_recorded_max = prerun.data['energy'].max()

    #get all train/ho data sets ONCE so it's not redone for each (p,d,q)
    splits = []

    for pred_date in prerun.train_dates['date']:
        train_mask = (
            (all_data['time'] < pred_date - timedelta(days=1)) &
            (all_data['time'] >= pred_date - timedelta(days=10))
        )
        ho_mask = (
            (all_data['time'] >= pred_date - timedelta(days=1)) &
            (all_data['time'] < pred_date + timedelta(days=1))
        )

        train_data = all_data.loc[train_mask].reset_index(drop=True)
        ho_data = all_data.loc[ho_mask].reset_index(drop=True)

        splits.append((pred_date, train_data, ho_data))
    
    results_dict = {}
    # print(system_recorded_max)
    # print(ho_data)
    # print(ho_data.columns)

    for p,d,q in product(p_choices,d_choices,q_choices):
        print(f'     starting p={p}, d={d}, q={q}')
        errors=[]
        
        for pred_date, train_data, ho_data in splits:
            try: 
                y_pred = use_sarimax(train_data, ho_data, p=p, d=d, q=q)
                # print(f"original y_pred: {y_pred}")

                #make sure value between 0 and highest observed max
                y_pred = np.clip(y_pred, 0, system_recorded_max)
                #make sure that if it's not sunlight hours and irradiance = 0, then energy = 0
                darkness_mask = (~((ho_data['proportion_daytime']==0) & (ho_data['global_tilted_irradiance']==0))).astype(int) # = 0 when both sublight prop and irr are 0
                y_pred = y_pred*np.array(darkness_mask)
                # print(f"y_pred with insurance: {y_pred}")

                energy_ho = ho_data['energy']
                # print(f"energy_ho: {energy_ho}")
                
                error = PostRun.custom_error(energy_ho.iloc[24:], y_pred.iloc[24:], 1,2)
                # print(f'error = {error}')
            except Exception as e:
                print(f'FAILED: (p,d,q) = {p,d,q} --> {type(e).__name__}: {e}')
                error = -1
                

            errors.append(error)

        results_dict[f"{p},{d},{q}"] = errors
    all_errors_df = pd.DataFrame(results_dict)
    all_errors_df.to_csv(f'sarimax_errors/{system_id}_{reader_type}_sarimax_errors.csv', index = False)

starting system 50, None
     starting p=2, d=0, q=0


c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed t

     starting p=2, d=0, q=1


c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parame

     starting p=3, d=0, q=0


...

c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed t

     starting p=3, d=0, q=0


c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed t

     starting p=3, d=0, q=1


c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed t

In [4]:
# #compare hyperparameters
# #System 4
# print('System 4, None, SARIMAX')
# errors = pd.read_csv('sarimax_errors/4_None_sarimax_errors.csv')
# hyperparams = errors.columns

# prerun = PreRun(system_id = 4, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
# prerun.load_data()
# system_recorded_max = prerun.data['energy'].max()
# print(f'recorded system max: {system_recorded_max}')

# for hp in hyperparams:
#     err = errors[hp]
#     print(f'  Hyperparameters: {hp}')
#     print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
# #all seem to be fine. Leaning toward 3,0,1. Then 3,0,0. Then 2,0,0

In [ ]:
#compare hyperparameters
#System 10
print('System 10, None, SARIMAX')
errors = pd.read_csv('sarimax_errors/10_None_sarimax_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 10, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
#al very close
#2,0,0; 3,0,0; 2,0,1; 3,0,1
#NOTE: ran with d=1 before and was significantly worse. Narrowed down to d=0.

System 10, None, SARIMAX
recorded system max: 1.185825
  Hyperparameters: 2,0,0
     mean: 0.02547952055322282, median: 0.012818421436861599, min: 0.0002591027718161, max: 0.1332870291469238, std: 0.03248074901110252
  Hyperparameters: 2,0,1
     mean: 0.025889175669852937, median: 0.01376268694350275, min: 0.000239998216452, max: 0.1357527918507043, std: 0.03342535261781394
  Hyperparameters: 3,0,0
     mean: 0.025593822129341862, median: 0.0126956120188789, min: 0.0002397735075866, max: 0.1357022721874045, std: 0.03281076135113786
  Hyperparameters: 3,0,1
     mean: 0.025968068223184856, median: 0.0144355785515389, min: 0.000240000265541, max: 0.1357572256981232, std: 0.03269300138721142


In [6]:
# #compare hyperparameters
# #System 33
# print('System 33, None, SARIMAX')
# errors = pd.read_csv('sarimax_errors/33_None_sarimax_errors.csv')
# hyperparams = errors.columns

# prerun = PreRun(system_id = 33, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
# prerun.load_data()
# system_recorded_max = prerun.data['energy'].max()
# print(f'recorded system max: {system_recorded_max}')

# for hp in hyperparams:
#     err = errors[hp]
#     print(f'  Hyperparameters: {hp}')
#     print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
# #several ok ones. Max's are getting higher though.
# #3,0,0; 2,0,0; 3,0,1; 2,0,1

In [ ]:
#compare hyperparameters
#System 50
print('System 50, None, SARIMAX')
errors = pd.read_csv('sarimax_errors/50_None_sarimax_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 50, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
#3,0,0; 3,0,1; 2,0,0; 2,0,1
#prefer 3,0,0 due to fewer errors

System 50, None, SARIMAX
recorded system max: 7.072975
  Hyperparameters: 2,0,0
     mean: 0.42116902986489047, median: 0.2447224093483765, min: 9.573573035474448e-12, max: 4.65270696866, std: 0.5745402737215851
  Hyperparameters: 2,0,1
     mean: 0.4304010868073856, median: 0.25262891691158207, min: 9.573573035099763e-12, max: 4.634019304346637, std: 0.580941692820293
  Hyperparameters: 3,0,0
     mean: 0.4169801368508425, median: 0.2276994828286624, min: 9.573573035390575e-12, max: 4.626920578829719, std: 0.5714103629332746
  Hyperparameters: 3,0,1
     mean: 0.41341298245285213, median: 0.22226530819927076, min: 9.573573035600666e-12, max: 4.636143632384246, std: 0.5725500302131522


In [ ]:
#compare hyperparameters
#System 51
print('System 51, None, SARIMAX')
errors = pd.read_csv('sarimax_errors/51_None_sarimax_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 51, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )

#much less clear. There are some clear "lowest mean", but some others seem to have a much lower spread.
# Preferred because of mean: 3,0,0; 3,0,1; followed by 2,0,0 and 2,0,1


System 51, None, SARIMAX
recorded system max: 7.2368749999999995
  Hyperparameters: 2,0,0
     mean: 0.5804529658908447, median: 0.2863512726093225, min: 0.0116106245750907, max: 4.842273684864219, std: 0.8082385848215345
  Hyperparameters: 2,0,1
     mean: 0.5862060873510093, median: 0.29068488418277, min: 0.0220002576337766, max: 4.815697303376325, std: 0.8024569750587127
  Hyperparameters: 3,0,0
     mean: 0.5707964621453575, median: 0.2797114798536715, min: 0.0118955974833283, max: 4.80576169073969, std: 0.8066162358221716
  Hyperparameters: 3,0,1
     mean: 0.5696627014415505, median: 0.2801374989249591, min: 0.0118847999594964, max: 4.81746506312448, std: 0.8086121727473509


In [9]:
# #compare hyperparameters
# #System 1283
# print('System 1283, Inverter, SARIMAX')
# errors = pd.read_csv('sarimax_errors/1283_inverter_sarimax_errors.csv')
# hyperparams = errors.columns

# prerun = PreRun(system_id = 1283, meter_or_inverter = 'inverter', path=read_path, systems_cleaned=systems_cleaned)
# prerun.load_data()
# system_recorded_max = prerun.data['energy'].max()
# print(f'recorded system max: {system_recorded_max}')

# for hp in hyperparams:
#     err = errors[hp]
#     print(f'  Hyperparameters: {hp}')
#     print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
# #again the main 4 are really similar. 2,0,0/3,0,0 have slightly lower means. 2,0,1/3,0,1 have slightly lower max's and std

In [10]:
# #compare hyperparameters
# #System 1283
# print('System 1283, Meter, SARIMAX')
# errors = pd.read_csv('sarimax_errors/1283_meter_sarimax_errors.csv')
# hyperparams = errors.columns

# prerun = PreRun(system_id = 1283, meter_or_inverter = 'meter', path=read_path, systems_cleaned=systems_cleaned)
# prerun.load_data()
# system_recorded_max = prerun.data['energy'].max()
# print(f'recorded system max: {system_recorded_max}')

# for hp in hyperparams:
#     err = errors[hp]
#     print(f'  Hyperparameters: {hp}')
#     print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
# #again the main 4 are really similar.
# # 3,0,0; 2,0,1; 2,0,0; 3,0,1

Run winning parameters ONLY for test set!

In [11]:
system_reader_pairs = [(10,None), (50,None), (51,None)]


all_errors = []
for pair in system_reader_pairs:
    system_id = pair[0]
    reader_type = pair[1]
    print(f'starting system {system_id}, {reader_type}')

    prerun = PreRun(system_id = system_id, meter_or_inverter = reader_type, path=read_path, systems_cleaned=systems_cleaned)
    prerun.load_data()
        
    prerun.fill_missing_hours()
    prerun.add_energy_features_only(include_last_year = True, remove_last_year_nans=True, highest_fourier_term_hour=2)
    #print(prerun.amended_data)

    prerun.add_weather_features_only()

    all_data=prerun.amended_data

    prerun.good_end_days_naive(10)
    prerun.tts_of_data_using_end_days(remove_first_year=False)

    system_recorded_max = prerun.data['energy'].max()

    splits = []

    for pred_date in prerun.test_dates['date']:
        train_mask = (
            (all_data['time'] < pred_date - timedelta(days=1)) &
            (all_data['time'] >= pred_date - timedelta(days=10))
        )
        ho_mask = (
            (all_data['time'] >= pred_date - timedelta(days=1)) &
            (all_data['time'] < pred_date + timedelta(days=1))
        )

        train_data = all_data.loc[train_mask].reset_index(drop=True)
        ho_data = all_data.loc[ho_mask].reset_index(drop=True)

        splits.append((pred_date, train_data, ho_data))
    
    # print(system_recorded_max)
    # print(ho_data)
    # print(ho_data.columns)

    #choose p,d,q
    d=0
    q=0
    if system_id == 10:
        p=2
    elif (system_id == 50) | (system_id == 51):
        p=3

    
    system_errors=[]  
    for pred_date, train_data, ho_data in splits:
        try: 
            y_pred = use_sarimax(train_data, ho_data, p=p, d=d, q=q)
            # print(f"original y_pred: {y_pred}")

            #make sure value between 0 and highest observed max
            y_pred = np.clip(y_pred, 0, system_recorded_max)
            #make sure that if it's not sunlight hours and irradiance = 0, then energy = 0
            darkness_mask = (~((ho_data['proportion_daytime']==0) & (ho_data['global_tilted_irradiance']==0))).astype(int) # = 0 when both sublight prop and irr are 0
            y_pred = y_pred*np.array(darkness_mask)
            # print(f"y_pred with insurance: {y_pred}")

            energy_ho = ho_data['energy']
            # print(f"energy_ho: {energy_ho}")
            
            error = PostRun.custom_error(energy_ho.iloc[24:], y_pred.iloc[24:], 1,2)
            # print(f'error = {error}')
        except Exception as e:
            print(f'FAILED: (p,d,q) = {p,d,q} --> {type(e).__name__}: {e}')
            error = -1
        system_errors.append(error)
    system_errors_df = pd.DataFrame({'error':system_errors})
    system_errors_df.to_csv(f'sarimax_errors/{system_id}_test_set_sarimax_errors.csv', index = False)
                



starting system 10, None


c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed t

starting system 50, None


c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed t

starting system 51, None


c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed t

look at errors!

In [15]:
systems = [10,50,51]

for system in systems:
    df = pd.read_csv(f'sarimax_errors/{system}_test_set_sarimax_errors.csv')
    print('System',system)
    print(f'mean = {df.mean()}, median = {df.median()}, min = {df.min()}, max = {df.max()}, std = {df.std()}')
    

System 10
mean = error    0.023272
dtype: float64, median = error    0.006188
dtype: float64, min = error    0.000908
dtype: float64, max = error    0.087627
dtype: float64, std = error    0.034465
dtype: float64
System 50
mean = error    0.312543
dtype: float64, median = error    0.158886
dtype: float64, min = error    0.035095
dtype: float64, max = error    1.949484
dtype: float64, std = error    0.42883
dtype: float64
System 51
mean = error    0.413511
dtype: float64, median = error    0.165404
dtype: float64, min = error    0.040071
dtype: float64, max = error    2.550656
dtype: float64, std = error    0.591266
dtype: float64
